#### **A: Introduction to Notebooks**

We will run all of the demos and exercises for this course in notebooks. Notebooks ar interactive documents that combine live code, visualizations, narrative text, and outputs in a sigle document. In Databricks, notebooks provide a powerful environment for data exploration, analysis, and collaboration.

Key features of Databricks notebooks:

- Code execution: RUn code in multiple languages (Python, SQL, R, Scala)
- Rich text formatting: Support for Markdown to create well-documented analyses
- Cell-based structure: Code and content is organized into executable cells
- Interactive visualizations: Direct plotting and charting of results
- Collaboration: Share and work together on notebooks in real-time

The default language for a notebook is set when you create it, as indicated by the **Python** selection in the toolbar above . You can change the default language through the notebook settings. Additionally, you can use magic commands to execute code in different languages within the same notebook:

* Use `%python`to execute Python code
* Use `%sql`to execute SQL queries
* Use `%R`to execute R code
* Use `%scala`to execute Scala code

The Use `%run` magic command is particularly useful - it allows you to execute another notebook within your current notebook, enabling modular code organization and reuse. For example `%run /path/to/another/notebook`.

This will execute all the cells in the referenced notebook as if they were part of your current notebook.

#### **B. The SparkSession and SparkContext**

The SparkSession is automatically instantiated as `spark`in Databricks notebook connected to a cluster. The SparkSession is available via the SpartSession usgin `spark.sparkCotext` or simply `sc` .

In [ ]:
from pyspark.sql import SparkSession

# Create a the first SparkSession
spark = SparkSession.builder.appName("FirstSession").getOrCreate()

# Create a new SparkSession using NewSession
# spark2 = spark.newSession()

# Both sessions will have the same SparkContext
# print("SparkContext is the same:", spark.sparkContext is spark2.sparkContext)

#### **C. Createing and Monitoring a Spark Job**

Let's create a simple job that will help us visualize the execution flow. Run the below cell to create a spark Job

**Note:** Don't focus too much on the code here. We want to focus on exploring the access logs and metrics.

In [ ]:
# Create a large DataFrame to see parallelization in action

from pyspark.sql.functions import *
import time

# Generate some data
df = spark.range(0, 1000000)
df = df.withColumn("squared", col("id") * col("id"))

# Force multiple stages with a shuffle operation
result = df.groupBy((col("id") % 100).alias("modulo")).agg(sum("squared").alias("sum_squared"))

# ressult = df.repartition(2)
# Cache the result to see storage in the UI
result.cache()

# Force the computation with the 'count' action
#print(f"Number of groups: {result.count()}")
result.collect()

In [ ]:
# Force the computation with the 'count' action
print(f"Number of groups: {result.count()}")

#### **Exploring the Spark UI**

Now that we have an active job, let's explore key areas of the Spark UI:

1. **Jobs Tab**
    - Shows that DAG for our groupBy operation
    - Multiple stages due to the shuffle operation
    - Click on a stage to see task-level details

2. **Executors Tab**
    - Lists all executors and their resource usage
    - Shows how many cores and memory each executor has
    - Demonstrates the Worker node distribution

3. **Storage Tab**
    - Shows cached DataFrames
    - Displays memory usage across executors

Notice how the architecture we discussed (Driver -> Master -> Workers -> Executors) is reflected i the UI. The Driver coordinates the job, while Executors on Worker nodes perform the actual computations (although in this class all processes reside on a sible node).



In [ ]:
# Free up executor memory by unpersisting cached objects.
result.unpersist() 
